# ЛР2. Отчёт по популярности языков программирования в Apache Spark

По постам Stack Overflow считаются 10 самых частых языков программирования для каждого года.

## 1. Установка PySpark

In [1]:
%pip install pyspark

## 2. SparkSession

In [2]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("LR2 Top Programming Languages")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## 3. Загрузка данных

In [3]:
from pathlib import Path


def find_file(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return None

posts_path = find_file([
    "../data/posts_sample.xml",
    "data/posts_sample.xml",
    "/content/posts_sample.xml",
    "posts_sample.xml",
])
languages_path = find_file([
    "../data/programming-languages.csv",
    "data/programming-languages.csv",
    "/content/programming-languages.csv",
    "programming-languages.csv",
])

if posts_path is None or languages_path is None:
    try:
        from google.colab import files
        print("Upload posts_sample.xml and programming-languages.csv")
        files.upload()
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            "posts_sample.xml and programming-languages.csv were not found. Put them into data/ or near the notebook."
        ) from exc

    posts_path = find_file(["/content/posts_sample.xml", "posts_sample.xml"])
    languages_path = find_file(["/content/programming-languages.csv", "programming-languages.csv"])

if posts_path is None or languages_path is None:
    raise FileNotFoundError("Input files were not found after upload")

print(f"posts: {posts_path}")
print(f"languages: {languages_path}")

posts_raw = spark.read.text(str(posts_path))
languages_raw = spark.read.csv(str(languages_path), header=True, inferSchema=True)

Upload posts_sample.xml and programming-languages.csv


Saving posts_sample.xml to posts_sample.xml
Saving programming-languages.csv to programming-languages.csv
posts: /content/posts_sample.xml
languages: /content/programming-languages.csv


## 4. Первичная проверка

In [4]:
posts_raw.printSchema()
languages_raw.printSchema()

print("raw XML rows:", posts_raw.count())
print("languages rows:", languages_raw.count())

posts_raw.show(5, truncate=False)
languages_raw.show(10, truncate=False)

root
 |-- value: string (nullable = true)

root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)

raw XML rows: 46009
languages rows: 700
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 5. Подготовка постов Stack Overflow

In [5]:
def attr(name):
    return F.regexp_extract(F.col("value"), rf'{name}="([^"]*)"', 1)

posts = (
    posts_raw
    .filter(F.col("value").contains("<row "))
    .select(
        attr("Id").cast("long").alias("post_id"),
        attr("PostTypeId").cast("int").alias("post_type_id"),
        F.to_timestamp(attr("CreationDate")).alias("creation_ts"),
        attr("Tags").alias("tags_raw"),
    )
    .filter(F.col("post_type_id") == 1)
    .filter(F.col("creation_ts").isNotNull())
    .withColumn("year", F.year("creation_ts"))
    .filter(F.col("year").between(2010, 2020))
    .filter(F.col("tags_raw") != "")
)

print("questions 2010-2020 with tags:", posts.count())
posts.printSchema()
posts.show(10, truncate=False)

questions 2010-2020 with tags: 17642
root
 |-- post_id: long (nullable = true)
 |-- post_type_id: integer (nullable = true)
 |-- creation_ts: timestamp (nullable = true)
 |-- tags_raw: string (nullable = true)
 |-- year: integer (nullable = true)

+-------+------------+-----------------------+--------------------------------------------------------------+----+
|post_id|post_type_id|creation_ts            |tags_raw                                                      |year|
+-------+------------+-----------------------+--------------------------------------------------------------+----+
|3768363|1           |2010-09-22 10:33:21.79 |&lt;c++&gt;&lt;character-encoding&gt;                         |2010|
|3775996|1           |2010-09-23 06:47:28.92 |&lt;sharepoint&gt;&lt;infopath&gt;                            |2010|
|3776721|1           |2010-09-23 08:53:14.017|&lt;iphone&gt;&lt;app-store&gt;&lt;in-app-purchase&gt;        |2010|
|3777993|1           |2010-09-23 11:47:00.833|&lt;symfony1&gt;

## 6. Проверка качества данных

In [6]:
posts.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in ["post_id", "post_type_id", "creation_ts", "tags_raw", "year"]
]).show(truncate=False)

posts.groupBy("year").count().orderBy("year").show(20, truncate=False)

+-------+------------+-----------+--------+----+
|post_id|post_type_id|creation_ts|tags_raw|year|
+-------+------------+-----------+--------+----+
|0      |0           |0          |0       |0   |
+-------+------------+-----------+--------+----+

+----+-----+
|year|count|
+----+-----+
|2010|712  |
|2011|1172 |
|2012|1677 |
|2013|2082 |
|2014|2112 |
|2015|2172 |
|2016|2190 |
|2017|2076 |
|2018|2026 |
|2019|1423 |
+----+-----+



## 7. Теги постов и справочник языков

In [7]:
posts_with_tags = (
    posts
    .withColumn(
        "tags_clean",
        F.regexp_replace(
            F.regexp_replace(F.col("tags_raw"), "^&lt;|&gt;$", ""),
            "&gt;&lt;",
            ",",
        ),
    )
    .withColumn("tag", F.explode(F.split(F.lower(F.col("tags_clean")), ",")))
    .select("post_id", "year", "tag")
    .filter(F.col("tag") != "")
)

languages = (
    languages_raw
    .select(F.trim(F.col("name")).alias("language"))
    .filter(F.col("language").isNotNull())
    .filter(F.col("language") != "")
)

language_tags = (
    languages.select(
        "language",
        F.lower(F.regexp_replace(F.col("language"), r"\s+", "-")).alias("tag"),
    )
    .unionByName(languages.select("language", F.lower(F.col("language")).alias("tag")))
    .dropDuplicates(["tag"])
)

print("exploded tags:", posts_with_tags.count())
print("languages:", languages.count())
print("language aliases:", language_tags.count())

posts_with_tags.show(20, truncate=False)
language_tags.orderBy("tag").show(30, truncate=False)

exploded tags: 52118
languages: 700
language aliases: 819
+-------+----+------------------+
|post_id|year|tag               |
+-------+----+------------------+
|3768363|2010|c++               |
|3768363|2010|character-encoding|
|3775996|2010|sharepoint        |
|3775996|2010|infopath          |
|3776721|2010|iphone            |
|3776721|2010|app-store         |
|3776721|2010|in-app-purchase   |
|3777993|2010|symfony1          |
|3777993|2010|schema            |
|3777993|2010|doctrine          |
|3777993|2010|fixtures          |
|3778222|2010|java              |
|3785460|2010|visual-studio-2010|
|3785460|2010|stylecop          |
|3789116|2010|cakephp           |
|3789116|2010|file-upload       |
|3789116|2010|swfupload         |
|3794815|2010|git               |
|3794815|2010|cygwin            |
|3794815|2010|putty             |
+-------+----+------------------+
only showing top 20 rows
+------------+------------+
|language    |tag         |
+------------+------------+
|@Formula    |@fo

## 8. Top-10 языков по годам

In [8]:
language_counts = (
    posts_with_tags
    .join(F.broadcast(language_tags), on="tag", how="inner")
    .groupBy("year", "language")
    .agg(F.countDistinct("post_id").alias("posts_count"))
)

rank_window = Window.partitionBy("year").orderBy(
    F.desc("posts_count"),
    F.asc("language"),
)

report = (
    language_counts
    .withColumn("rank", F.row_number().over(rank_window))
    .filter(F.col("rank") <= 10)
    .select("year", "rank", "language", "posts_count")
    .orderBy("year", "rank")
)

report.groupBy("year").count().orderBy("year").show(20, truncate=False)
report.show(120, truncate=False)

+----+-----+
|year|count|
+----+-----+
|2010|10   |
|2011|10   |
|2012|10   |
|2013|10   |
|2014|10   |
|2015|10   |
|2016|10   |
|2017|10   |
|2018|10   |
|2019|10   |
+----+-----+

+----+----+-----------+-----------+
|year|rank|language   |posts_count|
+----+----+-----------+-----------+
|2010|1   |Java       |52         |
|2010|2   |PHP        |46         |
|2010|3   |JavaScript |44         |
|2010|4   |Python     |26         |
|2010|5   |Objective-C|23         |
|2010|6   |C          |20         |
|2010|7   |Ruby       |12         |
|2010|8   |Delphi     |8          |
|2010|9   |AppleScript|3          |
|2010|10  |Bash       |3          |
|2011|1   |PHP        |102        |
|2011|2   |Java       |93         |
|2011|3   |JavaScript |83         |
|2011|4   |Python     |37         |
|2011|5   |Objective-C|34         |
|2011|6   |C          |24         |
|2011|7   |Ruby       |20         |
|2011|8   |Perl       |9          |
|2011|9   |Delphi     |8          |
|2011|10  |Bash       |7 

## 9. Сохранение в Parquet

In [9]:
output_path = "lr2_top_languages_by_year.parquet"

(
    report.write
    .mode("overwrite")
    .partitionBy("year")
    .parquet(output_path)
)

print(f"saved to: {output_path}")

saved to: lr2_top_languages_by_year.parquet


## 10. Проверка сохранённого Parquet

In [10]:
saved_report = spark.read.parquet(output_path)

saved_report.printSchema()
saved_report.orderBy("year", "rank").show(120, truncate=False)

saved_report.groupBy("year").count().orderBy("year").show(20, truncate=False)

root
 |-- rank: integer (nullable = true)
 |-- language: string (nullable = true)
 |-- posts_count: long (nullable = true)
 |-- year: integer (nullable = true)

+----+-----------+-----------+----+
|rank|language   |posts_count|year|
+----+-----------+-----------+----+
|1   |Java       |52         |2010|
|2   |PHP        |46         |2010|
|3   |JavaScript |44         |2010|
|4   |Python     |26         |2010|
|5   |Objective-C|23         |2010|
|6   |C          |20         |2010|
|7   |Ruby       |12         |2010|
|8   |Delphi     |8          |2010|
|9   |AppleScript|3          |2010|
|10  |Bash       |3          |2010|
|1   |PHP        |102        |2011|
|2   |Java       |93         |2011|
|3   |JavaScript |83         |2011|
|4   |Python     |37         |2011|
|5   |Objective-C|34         |2011|
|6   |C          |24         |2011|
|7   |Ruby       |20         |2011|
|8   |Perl       |9          |2011|
|9   |Delphi     |8          |2011|
|10  |Bash       |7          |2011|
|1   |PHP  

## Вывод

Итоговая таблица сохранена в Parquet. В `posts_sample.xml` есть данные за 2010-2019 годы; для полного диапазона 2010-2020 нужен полный дамп Stack Overflow.